# SkillForge — Fine-tuning LLaMA 3.2 3B on SkillVault

**Goal:** Fine-tune `meta-llama/Llama-3.2-3B-Instruct` on the SkillVault dataset  
**Method:** QLoRA (4-bit quantization + LoRA adapters)  
**Runtime:** GPU → Runtime > Change runtime type > T4 GPU  
**Time:** ~1.5 hours on free Colab T4  

---
## Pipeline
```
Cell 1: Install packages
Cell 2: Login to HuggingFace
Cell 3: Load + inspect SkillVault dataset
Cell 4: Format into LLaMA chat template
Cell 5: Load LLaMA 3.2 3B in 4-bit (QLoRA)
Cell 6: Configure LoRA adapters
Cell 7: Train
Cell 8: Evaluate — compare base vs fine-tuned
Cell 9: Save + push to HuggingFace
```

In [ ]:
# ─────────────────────────────────────────────
# CELL 1 — Install Packages
# ─────────────────────────────────────────────
# Run this first. Takes ~3 minutes.

!pip install -q \
    transformers==4.47.0 \
    datasets \
    peft==0.13.2 \
    trl==0.12.2 \
    bitsandbytes==0.45.0 \
    accelerate \
    huggingface_hub \
    torch \
    wandb

print('✅ Packages installed')

In [ ]:
# ─────────────────────────────────────────────
# CELL 2 — Login to HuggingFace
# ─────────────────────────────────────────────
# You need:
#   1. HuggingFace account + token (huggingface.co/settings/tokens)
#   2. LLaMA 3.2 access approved at: huggingface.co/meta-llama/Llama-3.2-3B-Instruct
#      (approval is instant — just click Accept on the model page)

from huggingface_hub import login, notebook_login

# This opens a widget to paste your HF token
notebook_login()

In [ ]:
# ─────────────────────────────────────────────
# CELL 3 — Load SkillVault Dataset
# ─────────────────────────────────────────────

from datasets import load_dataset
import json

# ── CONFIG — change these to your values ──
HF_USERNAME    = "ekaansh"          # your HuggingFace username
DATASET_REPO   = "skillvault"       # your dataset repo name
MODEL_NAME     = "skillforge-llama-3.2-3b"  # name for your output model
# ──────────────────────────────────────────

DATASET_ID = f"{HF_USERNAME}/{DATASET_REPO}"
OUTPUT_MODEL_ID = f"{HF_USERNAME}/{MODEL_NAME}"

print(f"📥 Loading dataset: {DATASET_ID}")
dataset = load_dataset(DATASET_ID)

train_ds = dataset["train"]
val_ds   = dataset["test"]

print(f"\n📊 Dataset loaded:")
print(f"   Train samples : {len(train_ds)}")
print(f"   Val samples   : {len(val_ds)}")
print(f"   Columns       : {train_ds.column_names}")

# Inspect one sample
print("\n📌 Sample training pair:")
sample = train_ds[0]
print(f"  pair_type  : {sample['pair_type']}")
print(f"  instruction: {sample['instruction'][:100]}...")
print(f"  input      : {sample['input'][:100]}...")
print(f"  output     : {sample['output'][:150]}...")

In [ ]:
# ─────────────────────────────────────────────
# CELL 4 — Format into LLaMA 3.2 Chat Template
# ─────────────────────────────────────────────
# LLaMA 3.2 Instruct uses this format:
#
#   <|begin_of_text|>
#   <|start_header_id|>system<|end_header_id|>
#   You are SkillForge...<|eot_id|>
#   <|start_header_id|>user<|end_header_id|>
#   Generate a SKILL.md for...<|eot_id|>
#   <|start_header_id|>assistant<|end_header_id|>
#   ---\nname: ...<|eot_id|>

SYSTEM_PROMPT = """You are SkillForge, an expert AI that generates professional \
agent skill files (SKILL.md). You deeply understand how AI coding agents like \
Claude Code, Cursor, Gemini CLI, and Codex use skill files to perform specialized \
tasks. You produce complete, accurate, and well-structured SKILL.md files with \
proper YAML frontmatter, clear instructions, trigger conditions, and examples."""


def format_prompt(sample):
    """
    Convert a training pair dict into a fully formatted LLaMA 3.2 prompt.
    The 'text' field is what the model trains on.
    """
    instruction = sample["instruction"]
    user_input  = sample["input"]
    output      = sample["output"]

    # Build user message
    user_msg = instruction
    if user_input and user_input.strip():
        user_msg = f"{instruction}\n\n{user_input}"

    # LLaMA 3.2 Instruct chat template
    text = (
        f"<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n"
        f"{SYSTEM_PROMPT}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n"
        f"{user_msg}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n"
        f"{output}<|eot_id|>"
    )

    return {"text": text}


# Apply formatting to both splits
train_formatted = train_ds.map(format_prompt, remove_columns=train_ds.column_names)
val_formatted   = val_ds.map(format_prompt,   remove_columns=val_ds.column_names)

print("✅ Dataset formatted")
print(f"\n📌 Sample formatted prompt (first 600 chars):")
print(train_formatted[0]["text"][:600])
print("...")

In [ ]:
# ─────────────────────────────────────────────
# CELL 5 — Load LLaMA 3.2 3B in 4-bit (QLoRA)
# ─────────────────────────────────────────────
# 4-bit quantization reduces model from ~6GB → ~2GB
# Leaves plenty of VRAM for training on T4 (15GB)

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"

# Check GPU
print(f"🖥️  GPU: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── 4-bit Quantization Config ──
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                        # load model in 4-bit
    bnb_4bit_quant_type="nf4",                # NormalFloat4 — best quality
    bnb_4bit_compute_dtype=torch.bfloat16,    # compute in bfloat16
    bnb_4bit_use_double_quant=True,           # double quantization saves more memory
)

print(f"\n📥 Loading {BASE_MODEL} in 4-bit...")

# ── Load Tokenizer ──
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token    # LLaMA has no pad token by default
tokenizer.padding_side = "right"             # pad right for training

# ── Load Model ──
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",                        # auto-place on GPU
    torch_dtype=torch.bfloat16,
)

# Disable caching during training (saves memory)
model.config.use_cache = False
model.config.pretraining_tp = 1

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\n✅ Model loaded")
print(f"   Total parameters : {total_params:,} ({total_params/1e9:.2f}B)")
print(f"   VRAM used        : {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ─────────────────────────────────────────────
# CELL 6 — Configure LoRA Adapters
# ─────────────────────────────────────────────
# LoRA adds small trainable matrices (A and B) alongside
# the frozen base model weights.
#
# Only these adapter params get trained — ~1% of total.
# Everything else stays frozen.

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for QLoRA training
model = prepare_model_for_kbit_training(model)

# ── LoRA Config ──
lora_config = LoraConfig(
    r=16,                    # rank — higher = more capacity, more memory
                             # 16 is sweet spot for this dataset size
    lora_alpha=32,           # scaling factor (usually 2x rank)
    target_modules=[         # which attention layers to adapt
        "q_proj",            # query projection
        "k_proj",            # key projection
        "v_proj",            # value projection
        "o_proj",            # output projection
        "gate_proj",         # feed-forward gate
        "up_proj",           # feed-forward up
        "down_proj",         # feed-forward down
    ],
    lora_dropout=0.05,       # slight dropout prevents overfitting
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

# Show trainable vs total
model.print_trainable_parameters()
# Expected: trainable params ~10-20M out of 3B total (~0.5%)

In [ ]:
# ─────────────────────────────────────────────
# CELL 7 — Train (FIXED v4 — version-proof)
# ─────────────────────────────────────────────

import inspect
from trl import SFTTrainer, SFTConfig

# ── Auto-detect what each class accepts ──
sft_config_params  = inspect.signature(SFTConfig.__init__).parameters
sft_trainer_params = inspect.signature(SFTTrainer.__init__).parameters

print("TRL version:", __import__('trl').__version__)
print("max_seq_length in SFTConfig :", "max_seq_length" in sft_config_params)
print("max_seq_length in SFTTrainer:", "max_seq_length" in sft_trainer_params)
print("processing_class in SFTTrainer:", "processing_class" in sft_trainer_params)
print("tokenizer in SFTTrainer      :", "tokenizer" in sft_trainer_params)

# ── Build SFTConfig kwargs ──
config_kwargs = dict(
    output_dir="/content/drive/MyDrive/skillforge-checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    optim="paged_adamw_32bit",
    weight_decay=0.001,
    warmup_steps=100,
    lr_scheduler_type="cosine",
    fp16=False,
    bf16=True,
    dataset_text_field="text",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    group_by_length=True,
    report_to="none",
)

# Add max_seq_length to SFTConfig only if it accepts it
if "max_seq_length" in sft_config_params:
    config_kwargs["max_seq_length"] = 2048
    print("✅ max_seq_length → SFTConfig")

training_args = SFTConfig(**config_kwargs)

# ── Build SFTTrainer kwargs ──
trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=train_formatted,
    eval_dataset=val_formatted,
)

# tokenizer vs processing_class
if "processing_class" in sft_trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
    print("✅ processing_class → SFTTrainer")
else:
    trainer_kwargs["tokenizer"] = tokenizer
    print("✅ tokenizer → SFTTrainer")

# max_seq_length in SFTTrainer only if it accepts it
if "max_seq_length" in sft_trainer_params:
    trainer_kwargs["max_seq_length"] = 2048
    print("✅ max_seq_length → SFTTrainer")

trainer = SFTTrainer(**trainer_kwargs)

print("\n🚀 Starting training...")
print(f"   Samples        : {len(train_formatted)}")
print(f"   Epochs         : {training_args.num_train_epochs}")
print(f"   Effective batch: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"   Learning rate  : {training_args.learning_rate}")

trainer.train(resume_from_checkpoint=True)
#trainer.train()
print("\n✅ Training complete!")

In [ ]:
# Run this in a NEW cell while Cell 7 is still running
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted — checkpoints will survive next disconnect")

In [ ]:
# ─────────────────────────────────────────────
# CELL 8 — Evaluate: Base vs Fine-tuned
# ─────────────────────────────────────────────
# Run the same prompt through the base model and
# your fine-tuned model. Compare outputs.

import textwrap

def generate_skill(prompt_text, max_new_tokens=512):
    """Generate a SKILL.md response for a given prompt."""
    full_prompt = (
        f"<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n"
        f"{SYSTEM_PROMPT}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n"
        f"{prompt_text}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n"
    )
    inputs = tokenizer(full_prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,       # low temperature = more deterministic
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    # Decode only the new tokens (not the prompt)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


# ── Test Prompts ──
test_prompts = [
    (
        "Generate a SKILL.md for working with Redis caching.",
        "Description: Best practices for Redis caching in Python apps\nAllowed tools: Bash, Read, Write"
    ),
    (
        "Generate a SKILL.md for Django REST API development.",
        "Description: Building and debugging Django REST Framework APIs\nAllowed tools: Bash, Read, Write, Edit"
    ),
    (
        "Generate a SKILL.md for Celery task queue management.",
        "Description: Creating, monitoring and debugging Celery tasks with Redis broker\nAllowed tools: Bash, Read"
    ),
]

print("📊 Evaluation — Fine-tuned Model Output")
print("=" * 60)

for i, (instruction, user_input) in enumerate(test_prompts, 1):
    prompt = f"{instruction}\n\n{user_input}"
    print(f"\n🧪 Test {i}: {instruction}")
    print("-" * 40)
    result = generate_skill(prompt)
    print(result[:800])  # Show first 800 chars
    print("..." if len(result) > 800 else "")
    print()

In [ ]:
# ─────────────────────────────────────────────
# CELL 9 — Save & Push to HuggingFace
# ─────────────────────────────────────────────

import os

SAVE_PATH = "./skillforge-final"

# ── Save locally ──
print("💾 Saving model locally...")
trainer.model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"   Saved → {SAVE_PATH}")

# List saved files
for f in os.listdir(SAVE_PATH):
    size = os.path.getsize(f"{SAVE_PATH}/{f}")
    print(f"   {f:40s} {size/1e6:.1f} MB")

# ── Push to HuggingFace ──
print(f"\n🚀 Pushing to HuggingFace: {OUTPUT_MODEL_ID}")

trainer.model.push_to_hub(
    OUTPUT_MODEL_ID,
    commit_message="SkillForge v1.0 — QLoRA fine-tuned on SkillVault dataset"
)
tokenizer.push_to_hub(
    OUTPUT_MODEL_ID,
    commit_message="Tokenizer for SkillForge v1.0"
)

print(f"\n✅ Model live at: https://huggingface.co/{OUTPUT_MODEL_ID}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 10 — (Optional) Merge LoRA into Base Model
# ─────────────────────────────────────────────
# Right now the model is saved as LoRA adapters only (~40MB).
# Merging produces a standalone model that runs without
# the PEFT library — better for deployment.

from peft import PeftModel
from transformers import AutoModelForCausalLM
import torch

print("🔀 Merging LoRA adapters into base model...")
print("   (This requires ~12GB RAM — skip if on free Colab)")

# Reload base in float16 (not 4-bit) for merging
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
)

# Load and merge LoRA weights
merged_model = PeftModel.from_pretrained(base_model, SAVE_PATH)
merged_model = merged_model.merge_and_unload()

MERGED_SAVE_PATH = "./skillforge-merged"
merged_model.save_pretrained(MERGED_SAVE_PATH, safe_serialization=True)
tokenizer.save_pretrained(MERGED_SAVE_PATH)

print(f"✅ Merged model saved → {MERGED_SAVE_PATH}")

# Push merged model
MERGED_MODEL_ID = f"{HF_USERNAME}/{MODEL_NAME}-merged"
merged_model.push_to_hub(MERGED_MODEL_ID)
tokenizer.push_to_hub(MERGED_MODEL_ID)

print(f"🚀 Merged model live: https://huggingface.co/{MERGED_MODEL_ID}")

---
## What to Watch During Training

| Metric | What It Means | Target |
|--------|--------------|--------|
| `train_loss` | How wrong the model is on training data | Should drop: 2.5 → 0.8 |
| `eval_loss` | How wrong on validation data | Should track train_loss |
| Gap between train/eval | If eval >> train → overfitting | Keep gap < 0.3 |
| `grad_norm` | Gradient size — spikes = instability | Should stay < 1.0 |

## Common Issues

**CUDA Out of Memory**  
→ Reduce `per_device_train_batch_size` from 2 to 1  
→ Increase `gradient_accumulation_steps` from 8 to 16  

**Loss not decreasing**  
→ Try learning rate 5e-4 instead of 2e-4  
→ Increase LoRA rank from 16 to 32  

**Loss decreasing then spiking**  
→ Reduce learning rate to 1e-4  
→ Increase warmup_ratio to 0.05  

**Output looks like base model (not domain-specific)**  
→ Run more epochs (3 → 5)  
→ Check dataset formatting in Cell 4  

---
## After Training

```python
# Load your model anywhere
from transformers import pipeline

pipe = pipeline("text-generation", model="ekaansh/skillforge-llama-3.2-3b")
result = pipe("Generate a SKILL.md for Django REST API development")
print(result[0]["generated_text"])
```